# Structure-aware paragraph splitting — prototype

**Verdict from `paragraph-split.ipynb`:** naive recursive-delimiter splitting at paragraph
scale cuts through functions, JSON subtrees, and prose paragraphs — sizes were fine, semantics
were not. This notebook prototypes the replacement.

**Approach name:** *structure-aware chunking*; for code specifically, *AST-based chunking* —
formalized in the cAST paper ("Enhancing Code RAG with Structural Chunking via Abstract
Syntax Tree"): parse the file into its structure tree, greedy-pack **whole sibling nodes**
into a paragraph until the budget is hit, recurse into any single node bigger than the
budget. A split can then only ever fall *between* complete structural units.

**Decisions carried in from the discussion:**
- structure is built **per file**, not per chunk — chunks later become packed runs of
  consecutive paragraphs (one boundary mechanism, perfect file → chunk → paragraph nesting)
- one tool family: **tree-sitter** grammars cover our code extensions AND markdown / json /
  yaml / toml / xml / html; files with no grammar (or no extension) get a synthetic
  delimiter-built structure (blank line -> line -> sentence -> comma -> word) via the same
  mechanism that deepens oversized structureless leaves inside parsed files
- paragraphs are byte spans of the raw file → content-hash identity and incremental updates
  work unchanged

**This notebook shows, for eyeball review:** (1) the structure tree each parser sees,
(2) the packed paragraphs, (3) parse+pack timing per file, extrapolated to corpus scale.

In [ ]:
import sqlite3
import time
from collections import defaultdict
from pathlib import Path
from statistics import mean, median

from tree_sitter_language_pack import get_parser

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"

BUDGET = 800  # bytes ≈ 200 tokens — paragraph size target

LANG_BY_EXT = {
    ".py": "python", ".java": "java", ".kt": "kotlin", ".js": "javascript",
    ".jsx": "javascript", ".ts": "typescript", ".tsx": "tsx", ".go": "go",
    ".rb": "ruby", ".rs": "rust", ".c": "c", ".h": "c", ".cpp": "cpp",
    ".hpp": "cpp", ".cs": "csharp", ".php": "php", ".sh": "bash", ".sql": "sql",
    ".md": "markdown", ".mdx": "markdown", ".html": "html",
    ".json": "json", ".yaml": "yaml", ".yml": "yaml", ".toml": "toml", ".xml": "xml",
}

_parsers = {}
def parser_for(path: Path):
    lang = LANG_BY_EXT.get(path.suffix.lower())
    if lang is None:
        return None, "plain"
    if lang not in _parsers:
        try:
            _parsers[lang] = get_parser(lang)
        except Exception:
            _parsers[lang] = None  # no grammar shipped -> plain-text fallback
    return _parsers[lang], (lang if _parsers[lang] else "plain")

## The packer — one rule over one tree

`pack_node` walks the structure tree: consecutive siblings accumulate into one paragraph
while they fit; a single oversized node is recursed into. When an oversized node has no
children (giant string literal, prose block, whole plain-text file), `deepen` CONTINUES
BUILDING THE TREE using text delimiters as synthetic structure — each delimiter is one
tree level (blank line -> line -> sentence -> comma -> word) — and the same packer keeps
walking. This is not the old split-then-merge recursion: delimiters only ever grow the
tree; all budgeting lives in the single packer. Byte-chop survives only as the true last
resort (a span with no spaces at all). A final pass merges structural crumbs (closing
braces, commas) into the preceding paragraph.

In [ ]:
DELIMS = [b"\n\n", b"\n", b". ", b",", b" "]
MIN_PARA = 64  # bytes; anything smaller merges into the preceding paragraph


SEP_NAMES = ["blank-line", "line", "sentence", "comma", "word"]


class Block:
    """Synthetic structure node for spans the parser gave no children for.
    label records which delimiter level produced it (for the outline view)."""
    def __init__(self, start, end, children=(), label="text"):
        self.start_byte, self.end_byte, self.children = start, end, list(children)
        self.label = label


def deepen(raw, start, end, level=0):
    """Text delimiters as structure BUILDERS: each delimiter adds one tree level
    inside an unstructured span; only still-oversized pieces deepen further. A
    level whose delimiter does not occur falls through to the next. Grows the
    tree only — budgeting/merging stays in pack_node."""
    if level >= len(DELIMS):
        return []
    sep = DELIMS[level]
    kids, pos = [], start
    for part in raw[start:end].split(sep):
        pend = pos + len(part)
        span_end = min(pend + len(sep), end)
        if part.strip():
            sub = deepen(raw, pos, span_end, level + 1) if span_end - pos > BUDGET else []
            kids.append(Block(pos, span_end, sub, label=SEP_NAMES[level]))
        pos = pend + len(sep)
    if len(kids) < 2:
        return deepen(raw, start, end, level + 1)
    return kids


def chop(start, end, budget):
    """True last resort: a span with no structure and no delimiters left."""
    return [(s, min(s + budget, end)) for s in range(start, end, budget)]


def pack_node(node, raw, budget):
    kids = list(node.children) or deepen(raw, node.start_byte, node.end_byte)
    if not kids:
        return chop(node.start_byte, node.end_byte, budget)
    spans, group_start = [], None
    for i, child in enumerate(kids):
        nxt_start = kids[i + 1].start_byte if i + 1 < len(kids) else node.end_byte
        if nxt_start - child.start_byte > budget:          # child alone busts the budget
            if group_start is not None:
                spans.append((group_start, child.start_byte))
                group_start = None
            spans += pack_node(child, raw, budget)
            if spans and spans[-1][1] < nxt_start:          # trailing gap rides with last sub-span
                spans[-1] = (spans[-1][0], nxt_start)
        elif group_start is None:
            group_start = child.start_byte
        elif nxt_start - group_start > budget:              # group would bust -> flush before child
            spans.append((group_start, child.start_byte))
            group_start = child.start_byte
    if group_start is not None:
        spans.append((group_start, node.end_byte))
    return spans


def merge_tiny(spans, min_size=MIN_PARA):
    """Structural crumbs (a lone brace, comma, import semicolon) merge into the
    preceding paragraph; a tiny first span merges forward. A slight budget
    overshoot beats a 3-byte paragraph."""
    out = []
    for s, e in spans:
        if out and e - s < min_size:
            out[-1] = (out[-1][0], e)
        else:
            out.append((s, e))
    if len(out) > 1 and out[0][1] - out[0][0] < min_size:
        out[0:2] = [(out[0][0], out[1][1])]
    return out


def structure_root(path: Path, raw: bytes):
    parser, fmt = parser_for(path)
    if parser:
        return parser.parse(raw).root_node, fmt
    return Block(0, len(raw), deepen(raw, 0, len(raw))), fmt


def split_file(path: Path):
    """Returns (format, parse_s, pack_s, spans) — spans are byte ranges of the raw file."""
    raw = path.read_bytes()
    t0 = time.monotonic()
    root, fmt = structure_root(path, raw)
    t1 = time.monotonic()
    if len(raw) <= BUDGET:
        spans = [(0, len(raw))]
    else:
        spans = merge_tiny([(s, e) for s, e in pack_node(root, raw, BUDGET) if raw[s:e].strip()])
    if spans:
        spans[0] = (0, spans[0][1])
    return fmt, t1 - t0, time.monotonic() - t1, spans

## Structure outlines — the packer's-eye view of each file

The COMPLETE tree as the packer walks it, per sample file. Reading guide:
- `[ts]` — node produced by an external parser (tree-sitter grammar: java/tsx/markdown/json/xml)
- `[syn]` — synthetic node built by OUR delimiter cascade (`deepen`), named after the
  delimiter level that produced it (`~blank-line`, `~line`, `~sentence`, `~comma`, `~word`)
- every node shows its byte size; nodes at or under BUDGET (800 B) are shown as leaves and
  NOT expanded — that is not truncation of the display: the packer treats a fitting node as
  an atom and packs it whole, so structure inside it never influences any boundary. Every
  node that IS bigger than the budget is expanded all the way down, through parser levels
  and synthetic levels alike — this is the full decision tree behind every split.
- LICENSE (no extension, no grammar) is all `[syn]` — the fallback in its pure form.

In [ ]:
SAMPLES = [
    "branow/webphone/server/src/main/java/com/scisbo/webphone/services/ContactService.java",
    "branow/webphone/client/src/pages/calls/CallPreviewControls.tsx",
    "branow/webphone/README.md",
    "branow/webphone/client/public/locales/uk/translation.json",
    "branow/webphone/server/pom.xml",
    "branow/webphone/LICENSE",
]
sample_paths = [GIT_CACHE / "github.com" / p for p in SAMPLES]
assert all(p.exists() for p in sample_paths)


def render_tree(node, raw, depth, out):
    is_syn = isinstance(node, Block)
    size = node.end_byte - node.start_byte
    name = f"~{node.label}" if is_syn else node.type
    preview = raw[node.start_byte:node.end_byte].split(b"\n")[0][:48].decode("utf-8", "replace")
    out.append(f"[{'syn' if is_syn else 'ts '}] {'  ' * depth}{name:<30} {size:>6} B  {preview}")
    if size > BUDGET:                       # a fitting node is a packing atom -> leaf in this view
        for child in list(node.children) or deepen(raw, node.start_byte, node.end_byte):
            render_tree(child, raw, depth + 1, out)
    return out


for path in sample_paths:
    raw = path.read_bytes()
    root, fmt = structure_root(path, raw)
    lines = render_tree(root, raw, 0, [])
    print("=" * 100)
    print(f"[{fmt}] {path.relative_to(GIT_CACHE)}  ({len(raw)} bytes, {len(lines)} packer-relevant nodes)")
    print("\n".join(lines))
    print()

## Packed paragraphs — full text, for eyeball review

The question to judge: is every paragraph a complete structural unit (whole method(s),
whole JSON subtree, whole prose paragraph(s)) — never a cut mid-unit?

In [ ]:
for path in sample_paths:
    raw = path.read_bytes()
    fmt, parse_s, pack_s, spans = split_file(path)
    sizes = sorted(e - s for s, e in spans)
    print("=" * 100)
    print(f"[{fmt}] {path.relative_to(GIT_CACHE)}")
    print(f"{len(raw)} bytes -> {len(spans)} paragraphs, sizes {sizes}")
    for i, (s, e) in enumerate(spans):
        text = raw[s:e].decode("utf-8", "replace")
        print(f"\n----- p{i}  ({e - s} bytes, ~{(e - s) // 4} tokens, bytes {s}-{e}) -----")
        print(text)
    print()

## Timing — parse + pack over every corpus file, extrapolated

tree-sitter is a compiled incremental parser; the hypothesis is that structure building is
noise next to embedding (75 ms/chunk even on the fast model). Measured over every distinct
file in the chunks table, then extrapolated.

In [ ]:
db = sqlite3.connect(DB_PATH)
files = [GIT_CACHE / inst / p for inst, p in
         db.execute("SELECT DISTINCT instance, path FROM chunks")]
db.close()

stats = defaultdict(list)
total_bytes = 0
t_all = time.monotonic()
for path in files:
    fmt, parse_s, pack_s, spans = split_file(path)
    stats[fmt].append((parse_s + pack_s, len(spans)))
    total_bytes += path.stat().st_size
t_all = time.monotonic() - t_all

n_paras = sum(n for v in stats.values() for _, n in v)
per_file = [t for v in stats.values() for t, _ in v]
print(f"{len(files)} files ({total_bytes / 1e6:.1f} MB) -> {n_paras} paragraphs "
      f"in {t_all:.2f}s wall\n")
header = f"{'format':<12} {'files':>6} {'mean ms':>8} {'median ms':>10} {'max ms':>7} {'paragraphs':>11}"
print(header); print("-" * len(header))
for fmt, v in sorted(stats.items(), key=lambda kv: -len(kv[1])):
    times = [t * 1000 for t, _ in v]
    print(f"{fmt:<12} {len(v):>6} {mean(times):>8.2f} {median(times):>10.2f} "
          f"{max(times):>7.1f} {sum(n for _, n in v):>11}")
print(f"\nmean {1000 * mean(per_file):.2f} ms/file -> extrapolation: "
      f"100k files ≈ {100_000 * mean(per_file) / 60:.1f} min, "
      f"1M files ≈ {1_000_000 * mean(per_file) / 3600:.1f} h (single core, no parallelism)")
print(f"for scale: embedding those {n_paras} paragraphs at 75 ms each ≈ "
      f"{n_paras * 0.075:.0f}s — structure building is {n_paras * 0.075 / t_all:.0f}x cheaper")

## Observations

(To fill in after eyeball review — evidence and verdict live together. Open items going in:
does TSX/JSX nesting recurse sensibly; do markdown sections keep their heading with their
body; does the JSON packer keep key context; is the plain-text fallback acceptable for
LICENSE-class files; paragraph→URI scheme decision once boundaries are settled.)